In [2]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

print("TensorFlow version:", tf.__version__)

# Build baseline LSTM model
model = Sequential([
    tf.keras.Input(shape=(78, 1)),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

TensorFlow version: 2.21.0


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,009 (74.25 KB)

 Trainable params: 19,009 (74.25 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# Load preprocessed LSTM data
X_train_lstm = np.load(PROCESSED_DIR / "X_train_lstm.npy")
X_val_lstm = np.load(PROCESSED_DIR / "X_val_lstm.npy")
X_test_lstm = np.load(PROCESSED_DIR / "X_test_lstm.npy")

y_train_lstm = np.load(PROCESSED_DIR / "y_train_lstm.npy")
y_val_lstm = np.load(PROCESSED_DIR / "y_val_lstm.npy")
y_test_lstm = np.load(PROCESSED_DIR / "y_test_lstm.npy")

print("Preprocessed data loaded successfully")
print("=" * 50)
print("Training   :", X_train_lstm.shape, y_train_lstm.shape)
print("Validation :", X_val_lstm.shape, y_val_lstm.shape)
print("Test       :", X_test_lstm.shape, y_test_lstm.shape)

Preprocessed data loaded successfully
Training   : (210000, 78, 1) (210000,)
Validation : (45000, 78, 1) (45000,)
Test       : (45000, 78, 1) (45000,)


In [7]:
# Train the baseline LSTM IDS

history = model.fit(
    X_train_lstm,
    y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=5,
    batch_size=256,
    verbose=1
)

Epoch 1/5
821/821 ━━━━━━━━━━━━━━━━━━━━ 138s 163ms/step - accuracy: 0.8066 - loss: 0.3940 - val_accuracy: 0.9230 - val_loss: 0.2128
Epoch 2/5
821/821 ━━━━━━━━━━━━━━━━━━━━ 130s 158ms/step - accuracy: 0.9255 - loss: 0.2053 - val_accuracy: 0.9319 - val_loss: 0.1771
Epoch 3/5
821/821 ━━━━━━━━━━━━━━━━━━━━ 124s 151ms/step - accuracy: 0.9345 - loss: 0.1691 - val_accuracy: 0.9397 - val_loss: 0.1519
Epoch 4/5
821/821 ━━━━━━━━━━━━━━━━━━━━ 145s 155ms/step - accuracy: 0.9436 - loss: 0.1463 - val_accuracy: 0.9456 - val_loss: 0.1387
Epoch 5/5
821/821 ━━━━━━━━━━━━━━━━━━━━ 128s 156ms/step - accuracy: 0.9451 - loss: 0.1412 - val_accuracy: 0.9506 - val_loss: 0.1171


In [8]:
# Evaluate baseline IDS on the clean test set

test_loss, test_accuracy = model.evaluate(
    X_test_lstm,
    y_test_lstm,
    verbose=1
)

print("\nClean Test Results")
print("=" * 50)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 24s 17ms/step - accuracy: 0.9504 - loss: 0.1160

Clean Test Results
Test Loss     : 0.1160
Test Accuracy : 0.9504
Test Accuracy : 95.04%


In [9]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Generate predictions on clean test data
y_test_prob = model.predict(X_test_lstm, verbose=1)
y_test_pred = (y_test_prob >= 0.5).astype(int).flatten()

# Confusion matrix
cm = confusion_matrix(y_test_lstm, y_test_pred)

print("Confusion Matrix")
print("=" * 50)
print(cm)

# Classification report
print("\nClassification Report")
print("=" * 50)
print(classification_report(
    y_test_lstm,
    y_test_pred,
    target_names=["BENIGN", "ATTACK"],
    digits=4
))

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 22s 16ms/step
Confusion Matrix
[[20974  1526]
 [  706 21794]]

Classification Report
              precision    recall  f1-score   support

      BENIGN     0.9674    0.9322    0.9495     22500
      ATTACK     0.9346    0.9686    0.9513     22500

    accuracy                         0.9504     45000
   macro avg     0.9510    0.9504    0.9504     45000
weighted avg     0.9510    0.9504    0.9504     45000



In [10]:
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.save(MODEL_DIR / "baseline_lstm.keras")

print("Baseline LSTM model saved successfully.")

Baseline LSTM model saved successfully.
